<a href="https://colab.research.google.com/github/dudl1/-/blob/main/Titans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install titans-pytorch

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from torch.cuda.amp import GradScaler, autocast
from titans_pytorch import MemoryAsContextTransformer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [3]:
model = MemoryAsContextTransformer(
    num_tokens=tokenizer.vocab_size, # Размер словаря
    dim=100, # Размерность эмбеддинга
    depth=3, # Глубина трансформера (количество слоев)
    segment_len=900, # Длина сегмента для локального внимания
    num_persist_mem_tokens=20, # Количество токенов постоянной памяти
    num_longterm_mem_tokens=400, # Количество токенов долговременной памяти
).cuda()

In [10]:
num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Количество обучаемых параметров: {num_parameters}")

Количество обучаемых параметров: 7341708


In [ ]:
print(model)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [5]:
'''
file_path = "/content/dataset.txt"
with open(file_path, "r") as file:
    text = file.read()

lines = text.splitlines()
tokens = tokenizer(" ".join(lines), return_tensors="pt", padding=True, truncation=True)
'''

file_path = "/content/dataset.txt"
with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

# Токенизация текста
tokens = tokenizer(
    lines,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=200
)

token_ids = tokens["input_ids"].cuda()
labels = token_ids.clone()

In [6]:
'''
batch_size = 1
seq_len = token_ids.size(1)
epochs = 30

# Цикл обучения
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    # Прогон через модель
    logits = model(token_ids)

    # Сдвиг токенов
    logits = logits[:, :-1, :].contiguous()  # Обрезаем последний временной шаг
    labels_shifted = labels[:, 1:].contiguous()  # Обрезаем первый временной шаг

    # Перекладываем в плоскую форму
    logits = logits.view(-1, logits.size(-1))  # (N, vocab_size)
    labels_shifted = labels_shifted.view(-1)   # (N,)

    # Вычисление потерь
    loss = criterion(logits, labels_shifted)
    loss.backward()
    optimizer.step()

    print(f"Эпоха {epoch + 1}/{epochs}, Потери: {loss.item()}")
'''

scaler = GradScaler()

# Параметры обучения
batch_size = 30  # Минимальный размер пакета для снижения нагрузки на память
seq_len = token_ids.size(1)
epochs = 4

# Цикл обучения
for epoch in range(epochs):
    model.train()
    total_loss = 0

    # Пошаговая обработка токенов (разделение на батчи)
    for i in range(0, token_ids.size(0), batch_size):
        # Подготовка батча
        batch_input = token_ids[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        optimizer.zero_grad()

        with autocast():  # Смешанная точность для экономии памяти
            # Прогон через модель
            logits = model(batch_input)

            # Сдвиг токенов
            logits = logits[:, :-1, :].contiguous()  # Обрезаем последний временной шаг
            labels_shifted = batch_labels[:, 1:].contiguous()  # Обрезаем первый временной шаг

            # Перекладываем в плоскую форму
            logits = logits.view(-1, logits.size(-1))  # (N, vocab_size)
            labels_shifted = labels_shifted.view(-1)   # (N,)

            # Вычисление потерь
            loss = criterion(logits, labels_shifted)

        # Обратное распространение с масштабированием
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Суммирование потерь
        total_loss += loss.item()

        # Очистка CUDA-кэша
        torch.cuda.empty_cache()

    # Вывод средней потери за эпоху
    avg_loss = total_loss / (token_ids.size(0) // batch_size)
    print(f"Эпоха {epoch + 1}/{epochs}, Средние потери: {avg_loss:.4f}")

<ipython-input-6-18121ecc747f>:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-6-18121ecc747f>:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # Смешанная точность для экономии памяти


Эпоха 1/4, Средние потери: 5.5978
Эпоха 2/4, Средние потери: 2.4544
Эпоха 3/4, Средние потери: 1.3226
Эпоха 4/4, Средние потери: 0.9273


In [181]:
def generate_text(model, tokenizer, prompt, max_length=150):
    model.eval()
    tokens = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
    generated = tokens

    for _ in range(max_length):
        logits = model(generated)[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat((generated, next_token), dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0])

# Генерация текста
prompt = '''
Second Citizen:
Your own desert!

CORIOLANUS:
Ay, but not mine own desire.
'''
generated_text = generate_text(model, tokenizer, prompt)
print("Сгенерированный текст:", generated_text)

Сгенерированный текст: [CLS] second citizen : your own desert! coriolanus : ay, but not mine own desire. [SEP], speak, speak. first citizen : you are all : you are all : resolved. first citizen : you. first, you. first citizen : resolved. all : he is, we know ' t, we know ' t, we ' t, we know ' t, we know ' ll have corn at our own price. is ' t a verdict? all : we ' t ; let us humanely. is. what authority surfeits on ' ll have corn at our own price. first citizen : we ' t a verdict? all : we. what authority surfeits on ' t. what authority surfeits on ' t. first citizen : the people. all : we ' ll have corn at our own price.
